# **Graph instance for Random Walk Interdiction Problem in Cybersecurity**

## **Introduction to Graph Databases with Neo4j**

This notebook relies on Neo4j, a native graph database designed to store, manage, and query highly connected data. Unlike traditional relational databases, Neo4j uses a property graph model (nodes, relationships, and properties) to represent complex data networks intuitively and efficiently.

In this notebook, we will:

- Establish a connection to a Neo4j database using the official Python driver.
- Use Cypher (Neo4j's query language) to create nodes and define relationships.
- Visualize the results of our queries directly within the notebook environment.

### **Documentation Links**
Here are the most useful links to get started:

- Main Neo4j Documentation: https://neo4j.com/docs/
- The Cypher Language Manual (essential): https://neo4j.com/docs/cypher-manual/current/
- Python Driver Documentation (for your notebook): https://neo4j.com/docs/python-manual/current/

### **Installation of Neo4j**

Neo4j is not just a Python library; it also requires installing the core database server as a Linux package. This involves fetching the official security key, adding the Neo4j repository to the system's list of software sources, and then downloading the package itself.

### **Configuration of Neo4j**
 We install the apoc plugin of Neo4j, this is the tool that allow export to json formats, we need also to configure the security of neo4j to accept it as an export tool

In [ ]:
%%bash
NEO4J_VERSION="5.18.0"
rm -rf neo4j_local
if [ ! -d "neo4j_local" ]; then
    wget -q -nc https://neo4j.com/artifact.php?name=neo4j-community-$NEO4J_VERSION-unix.tar.gz -O neo4j.tar.gz
    tar -xzf neo4j.tar.gz
    mv neo4j-community-$NEO4J_VERSION neo4j_local
    rm neo4j.tar.gz
fi
if [ ! -f "neo4j_local/plugins/apoc-$NEO4J_VERSION-core.jar" ]; then
    wget -q -nc https://github.com/neo4j/apoc/releases/download/$NEO4J_VERSION/apoc-$NEO4J_VERSION-core.jar -P neo4j_local/plugins/
fi
CONF_FILE="neo4j_local/conf/neo4j.conf"
APOC_CONF="neo4j_local/conf/apoc.conf"
if ! grep -q "dbms.security.procedures.unrestricted=apoc.\*" "$CONF_FILE"; then
    echo "dbms.security.procedures.unrestricted=apoc.*" >> "$CONF_FILE"
    echo "apoc.export.file.enabled=true" > "$APOC_CONF"
    ./neo4j_local/bin/neo4j-admin dbms set-initial-password "password"
fi
chmod -R 755 neo4j_local
echo "[+] Environnement Neo4j prêt !"

### **Configuration for notebook usage**

Now we move on to the configuration phase. Simply installing the database is not enough to use it within a notebook environment. We need adsimulator and all python code to generate and post process the graph database output, to do so we simply clone and init everything from our repository

In [ ]:
%%bash
# 1. Clean slate (Remove old repo and old adsimulator configs)
rm -rf Markov_Budget /root/.adsimulator

# 2. Clone and switch branches
git clone https://github.com/Maelh1/Markov_Budget
cd Markov_Budget
#git switch 3-viz-tool-and-tutorial #change with you want another branch

# 3. Setup submodules and LFS
git submodule update --init --recursive
git lfs pull
git submodule foreach git lfs pull

# 4. Install main repo requirements
pip install -r requirements.txt

In [ ]:
%%bash
cd Markov_Budget/adsimulator_graph_generator/adsimulator

echo "[*] Triggering the developer's root bypass..."
mkdir -p /root/.adsimulator
touch /root/.adsimulator/enable_root.cfg

echo "[*] Copying required data files..."
cp -r data /root/.adsimulator/
pip uninstall -y adsimulator
python setup.py install

In [ ]:
import shutil
import subprocess

# Let Python find the exact path dynamically (works locally and in Colab)
adsim_exe = shutil.which("adsimulator")

# Failsafe just in case Jupyter's cache is completely blind
if adsim_exe is None:
    adsim_exe = "/usr/local/bin/adsimulator"

**Important Note**: If the .py files are change, you nee to relaunch the kernel as the import function won't be change otherwise

In [ ]:
import sys
import os

# Get the absolute path to the repo
repo_path = os.path.abspath("./Markov_Budget/adsimulator_graph_generator")

# Add it to sys.path if it's not already there
if repo_path not in sys.path:
    sys.path.append(repo_path)

from src import adsim_utils
from src import viz_tools
import importlib
importlib.reload(adsim_utils)

import os
import json
import random
import networkx as nx
import numpy as np
import subprocess
import time
from neo4j import GraphDatabase
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
config = {
        "Domain": {
            "functionalLevelProbability": {
                "2008": 0, "2008 R2": 0, "2012": 0, "2012 R2": 0, "2016": 100, "Unknown": 0
            },
            "Trusts": {
                "SIDFilteringProbability": 100, "Inbound": 0, "Outbound": 0, "Bidirectional": 0
            }
        },
        "Computer": {
            "nComputers": 5, # Augmenté pour correspondre au nombre d'OUs
            "CanRDPFromUserPercentage": 0,
            "CanRDPFromGroupPercentage": 0,
            "CanPSRemoteFromUserPercentage": 0,
            "CanPSRemoteFromGroupPercentage": 0,
            "ExecuteDCOMFromUserPercentage": 0,
            "ExecuteDCOMFromGroupPercentage": 0,
            "AllowedToDelegateFromUserPercentage": 0,
            "AllowedToDelegateFromComputerPercentage": 0,
            "enabled": 100,
            "haslaps": 0,
            "unconstraineddelegation": 0,
            "osProbability": {
                "Windows XP Professional Service Pack 3": 0,
                "Windows 7 Professional Service Pack 1": 0,
                "Windows 7 Ultimate Service Pack 1": 0,
                "Windows 7 Enterprise Service Pack 1": 0,
                "Windows 10 Pro": 0,
                "Windows 10 Enterprise": 100
            }
        },
        "DC": {
            "enabled": 100,
            "haslaps": 0,
            "osProbability": {
                "Windows Server 2003 Enterprise Edition": 0,
                "Windows Server 2008 Standard": 0,
                "Windows Server 2008 Datacenter": 0,
                "Windows Server 2008 Enterprise": 0,
                "Windows Server 2008 R2 Standard": 0,
                "Windows Server 2008 R2 Datacenter": 0,
                "Windows Server 2008 R2 Enterprise": 0,
                "Windows Server 2012 Standard": 0,
                "Windows Server 2012 Datacenter": 0,
                "Windows Server 2012 R2 Standard": 0,
                "Windows Server 2012 R2 Datacenter": 0,
                "Windows Server 2016 Standard": 0,
                "Windows Server 2016 Datacenter": 100
            }
        },
        "User": {
            "nUsers": 5, # Augmenté légèrement par sécurité
            "enabled": 100, "dontreqpreauth": 0, "hasspn": 0, "passwordnotreqd": 0,
            "pwdneverexpires": 0, "sidhistory": 0, "unconstraineddelegation": 0
        },
        "OU": {
            "nOUs": 5 # Augmenté pour éviter la division par zéro
        },
        "Group": {
            "nGroups": 2,
            "nestingGroupProbability": 0,
            "departmentProbability": {
                "IT": 100, "HR": 0, "MARKETING": 0, "OPERATIONS": 0, "BIDNESS": 0
            }
        },
        "GPO": {
            "nGPOs": 1
        },
        "ACLs": {
            "ACLPrincipalsPercentage": 0,
            "ACLsProbability": {
                "GenericAll": 0,
                "GenericWrite": 0,
                "WriteOwner": 0,
                "WriteDacl": 0,
                "AddMember": 0,
                "ForceChangePassword": 0,
                "ReadLAPSPassword": 0
            }
        }
    }

adsim_utils.run_pipeline(0, custom_config=config)

## **Random Walk Interdiction Problem in Cybersecurity**

This notebook aims to propose a solution for generating graphs intended for cyberattack modeling.

### **Example 1: Building the default graph**

For the construction of the default graph, we tried to create the simplest possible graph that AdSimulator would allow. However, as we can see, it is far from simple, and the visualization of the graph doesn't really help.

The reason the graph is so complicated is that it contains a lot of default nodes. The idea is that when using Windows, user accounts are automatically created (e.g., Guest, Admin, etc.). Similarly, we are required to have a computer to host the AD server. And that is without even mentioning the groups.

Areas for improvement:
- Brainstorm a better way to represent the AD.
- Remove the default nodes?
- Remove nodes that provide no useful information?

In [ ]:
viz_tools.plot_ad_complete_graph('Dataset/graph_0.json')

### **Minimum number of users**

Despite our request (by default) to create only 5 users, we can see on the graph above that there are actually 9. This is perfectly normal, as there are 4 user accounts created by default on Windows, in addition to the user accounts assigned to actual people:
- Guest
- Default Account
- Admin
- KRBTGT

In [ ]:

compteur = 0
with open('Dataset/graph_0.json', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue

        data = json.loads(line)

        # On cherche uniquement les nœuds de type "User"
        if data.get('type') == 'node':
            labels = data.get('labels', [])
            if 'User' in labels:
                nom = data.get('properties', {}).get('name', 'Inconnu')
                print(f" 👤 {nom}")
                compteur += 1

print(f"\n[+] Total d'utilisateurs trouvés : {compteur}")

### **Minimum number of computers**

Same story here: we only requested the creation of 5 computers (by default), but we ended up with 8. In reality, there are also computers dedicated to:
- The Primary Domain Controller (PDC), which hosts the server. Its presence is mandatory.
- The other Domain Controllers (DC).

In [ ]:
print(f"[*] Recherche des ordinateurs dans : {'Dataset/graph_0.json'}...\n")

compteur = 0
with open('Dataset/graph_0.json', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue

        data = json.loads(line)

        # On cherche uniquement les nœuds de type "Computer"
        if data.get('type') == 'node':
            labels = data.get('labels', [])
            if 'Computer' in labels:
                nom = data.get('properties', {}).get('name', 'Inconnu')
                print(f" 💻 {nom}")
                compteur += 1

print(f"\n[+] Total d'ordinateurs trouvés : {compteur}")

### **Minimum number of groups**

Finally, the worst part involves the groups. Even though we only requested the creation of 2 groups (by defaut), there are actually dozens of them in reality.

In [ ]:
compteur = 0
with open('Dataset/graph_0.json', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue

        data = json.loads(line)

        # On cherche uniquement les nœuds ayant l'étiquette "Group"
        if data.get('type') == 'node':
            labels = data.get('labels', [])
            if 'Group' in labels:
                nom = data.get('properties', {}).get('name', 'Inconnu')
                print(f" 👥 {nom}")
                compteur += 1

print(f"\n[+] Total de groupes trouvés : {compteur}")

## **Complex Example: Building a graph with 20 users, 20 computers, 20 groups, and a 90% chance for a group to have maximum rights**

Here, the idea is to propose a more complex configuration to verify that we indeed retrieve many more attack paths than with the default setup. A complete configuration example follows below. Additionally, you can find in the appendix a detailed explanation of each parameter, along with its consequences on the graph generation.

In [ ]:
micro_ad_1 = {
    "Domain": {
        "functionalLevelProbability": {
            "2008": 4,
            "2008 R2": 5,
            "2012": 10,
            "2012 R2": 30,
            "2016": 50,
            "Unknown": 1
        },
        "Trusts": {
            "SIDFilteringProbability": 50,
            "Inbound": 3,
            "Outbound": 3,
            "Bidirectional": 4
        }
    },
    "Computer": {
        "nComputers": 20,
        "CanRDPFromUserPercentage": 80,
        "CanPSRemoteFromUserPercentage": 30,
        "enabled": 100,
        "osProbability": {"Windows 10 Enterprise": 100},
        "CanRDPFromGroupPercentage": 1,
        "CanPSRemoteFromGroupPercentage": 1,
        "ExecuteDCOMFromGroupPercentage": 1
    },
    "User": {
        "nUsers": 30,
        "enabled": 100,
        "hasspn": 20,
        "unconstraineddelegation": 15
    },
    "Group": {
        "nGroups": 10,
        "nestingGroupProbability": 50
    },
    "ACLs": {
        "ACLPrincipalsPercentage": 80,
        "ACLsProbability": {
            "GenericAll": 15,
            "AddMember": 40,
            "ForceChangePassword": 20,
            "WriteDacl": 10
        }
    }
}

adsim_utils.run_pipeline(1, custom_config=micro_ad_1)

## **Appendix: ADSimulator Configuration Parameters**
This document details the parameters used in the ADSimulator JSON configuration file. These values allow for the modulation of network size, topology, and the density of vulnerabilities (attack paths) generated.

### **Domain Parameters**

- `functionalLevelProbability`: Defines the probability distribution of Domain Functional Levels (from 2008 to 2016). This influences domain metadata. Older functional levels (e.g., 2008) can contextually justify the presence of legacy vulnerabilities.
- `Trusts`: Manages the number of domains given the expected relationships with other simulated domains :
  - `Inbound` : The number of domains trusted by the current domain.
  - `Outbound` : The number of domain that trust the current domain
  - `Bidirectional` : The number of domain that trust and are trusted by the current domain
  - `SIDFilteringProbability`: Defines the likelihood that SID Filtering is enabled. When filtering is disabled (low probability), AdSimulator generates critical attack edges (such as `MemberOf` or `AdminTo`) that cross domain boundaries.

### **Computer Parameters**
- `nComputers`: The number of standard computers generated.

**Access Permissions (User vs. Group)**:

- `CanRDPFrom[User/Group]Percentage`: The probability that a specific User or Group is granted RDP rights to a computer.
- `CanPSRemoteFrom[User/Group]Percentage`: The probability for WinRM/PowerShell Remoting access.
- `ExecuteDCOMFrom[User/Group]Percentage`: The probability for DCOM execution rights.
- **Graph Impact**: These generate CanRDP, CanPSRemote, and ExecuteDCOM edges. Distinguishing between users and groups allows the simulator to create more realistic "Admin Groups" (shorter paths) versus "Individual Over-privileged Users" (noisier paths).

**Delegation:**

- `AllowedToDelegateFrom[User/Computer]Percentage`: Defines the probability of Constrained Delegation. It creates AllowedToDelegate edges originating from either a User account or another Computer account.
- `unconstraineddelegation`: The probability that a computer is configured with Unconstrained Delegation (attribute unconstraineddelegation=true).
- **Graph Impact**: These are high-value targets. Unconstrained delegation allows an attacker to harvest TGTs from any high-privileged user (like a Domain Admin) who connects to that machine.

**Lifecycle & Security:**

- `enabled`: The percentage of computer objects that are "active." Inactive (disabled) computers still appear in the graph but might be filtered out during active attack path analysis.
- `haslaps`: The probability that LAPS is implemented. On the graph, this generates ReadLAPSPassword edges for designated administrative groups/users.
- `osProbability`: Defines the distribution of Operating Systems across the fleet.
- **Graph Impact**: While largely informational (metadata), older OS versions like Windows XP or Windows 7 contextually justify the existence of legacy vulnerabilities (like MS17-010) that an attacker could exploit to gain local admin rights without existing edges.

### **Domain Controller (DC) Parameters**

- `enabled`: The probability that the DC object is active.
- `osProbability`: Defines the distribution of Server Operating Systems specifically for DCs.
- `haslaps`: The probability that the Local Administrator Password Solution (LAPS) is applied to the DC.
  - **Graph Impact**: Creates ReadLAPSPassword edges. In a DC context, this is extremely sensitive as it gives local administrative access to the most protected server in the domain.


### **User Parameters (User)**

- `nUsers`: The total number of users to create.
- `enabled`: The percentage of active accounts.
- `dontreqpreauth`: Probability that an account does not require Kerberos pre-authentication. Makes the account vulnerable to AS-REP Roasting.
  - **Graph Impact**: Generates the `ASREPRoastable` property/edge. This allows an attacker to offline crack the user's password without sending any traffic to a workstation.
- `hasspn`: Probability that an account has a Service Principal Name (SPN).
  - **Graph Impact**: Generates the `HasSPN` property and makes the user Kerberoastable.
- `passwordnotreqd` : The probability that the account is configured to not require a password.
  - **Graph Impact**: While it doesn't create a new arrow (edge) automatically, it creates a "Zero-Cost" Node.
- `pwdneverexpires`: The percentage of accounts with non-expiring passwords.
  - **Graph Impact**: Increases the likelihood that old, leaked credentials found in breaches or memory (LSASS) are still valid.
- `sidhistory`: Probability that a user retains a SID history.
  - **Graph Impact**: Linked to the Trusts parameters. If SID Filtering is disabled, these users can escalate privileges across domain boundaries by the creation of links between a User in Domain A and a Group in Domain B.
- `unconstraineddelegation`: The probability that a User account is trusted for unconstrained delegation.
  - **Graph Impact**: Unlike the computer setting, this means if a high-privileged user interacts with a service running under this user's context, their TGT can be stolen.

### **Organizational Parameters (OU, Group, GPO)**
- `nOUs`, `nGroups`, `nGPOs`: Sets the quantity of these specific entities.

**For Groups only :**

- `nestingGroupProbability`: The probability that a group is a member of another group.
  - **Graph Impact**: Generates recursive MemberOf edges between Group nodes.
- `departmentProbability` : Assigns a specific Department Attribute to User nodes based on the defined distribution (IT, HR, Marketing, etc.).
  - **Graph Impact**: Primarily modifies node Properties (metadata).Tthis dictates the placement of Users into specific Organizational Units (OUs).

**Access Control Parameters (ACLs)**
- `ACLPrincipalsPercentage`: Percentage of users and groups involved in ACL rules.
- `ACLsProbability`: Distribution of specific rights assigned randomly:
  - `GenericAll` / `GenericWrite`: Full control or generic write rights over an object (immediate privilege escalation).
  - `WriteOwner` / `WriteDacl`: The right to take ownership of an object or modify its permissions.
  - `AddMember` : The right to add a member to a group (critical if the group is an admin group).
  - `ForceChangePassword`: The right to reset a user's password without knowing the current one.
  - `ReadLAPSPassword` : Provides the clear-text password of the local administrator.

### **Targeted visualization of attack paths**

To make the analysis clearer and more pleasant, we decided to change the way attack paths are displayed. Rather than overlapping all vulnerabilities on a single graph, which would quickly become cluttered and unreadable, we opted for an individual visualization approach. Displaying the attack paths one by one allows us to focus on one threat at a time, greatly facilitating the tracking and understanding of flaws within the network.

In [ ]:
with open("./Dataset/graph_1_structured.json", 'r') as f:
    graph_data = json.load(f)
num_nodes = graph_data['metadata']['nodes_count']
edges = graph_data['subgraph_topology']['edge_index']
node_registry = graph_data['node_registry']
T = adsim_utils.build_transition_matrix(edges, num_nodes)
# Extract best known allocation for colors
actual_alloc = np.zeros(num_nodes)
for i in range(num_nodes):
    actual_alloc[i] = node_registry[str(i)].get('best_allocation_weight', 0.0)

# 2. Automatically find ALL Sources and ALL Targets
sources = [int(node_id) for node_id, data in node_registry.items() if data.get('is_source') == True]
targets = [int(node_id) for node_id, data in node_registry.items() if data.get('is_terminal') == True]

print(f"Detected {len(sources)} attacker entry points (sources) and {len(targets)} critical assets (targets).")
print(f"Total possible direct threat vectors to evaluate: {len(sources) * len(targets)}\n")

# --- PART A: Tell the story of the whole graph ---
print("Generating the Macro View of the Network...")
viz_tools.plot_full_network(edges, num_nodes, sources, targets, actual_alloc, node_registry)

# --- PART B: Zoom in on the specific attack paths ---
for source in sources:
    for target in targets:
        print(f"\n--- Analyzing threat vector from Node {source} to Node {target} ---")

        viz_tools.plot_single_attack_path(
            edges=edges,
            num_nodes=num_nodes,
            source=source,
            target=target,
            allocation=actual_alloc,
            node_registry=node_registry,
            T=T
        )

## The Attacker's Dilemma: The Path of Least Resistance

In a real cyber intrusion, the network is rarely a straight line from A to B; it is a maze of interconnected systems, shared credentials, and vulnerable services.

When an attacker breaches a **Source** (🟩), they scan the environment. The algorithm might tell us the mathematically *shortest* path to the **Target** (🟥), but what if we (the defenders) have anticipated this? What if we placed all our defense budget (🟦 Dark Blue nodes) right in the middle of that shortest path?

Faced with heavy defenses (like multi-factor authentication, monitored jump servers, or strict firewalls), the attacker has a choice:
1. **The Direct Assault:** Try to brute-force or bypass the heavily defended chokepoint on the shortest path. This is noisy and has a low probability of success.
2. **The Scenic Route:** Pivot laterally. Find a secondary, slightly longer path through lightly defended, forgotten infrastructure (🟦 Light Blue nodes) to bypass the chokepoint entirely.

To truly understand our security posture, we cannot just look at a single path. We must look at the **alternative routes**. If our primary path is locked down, but a secondary path is wide open, our network is still highly vulnerable. Let's extract multiple paths to see how an attacker might pivot around our defenses.

In [ ]:
viz_tools.plot_multiple_attack_paths_clean(edges, num_nodes, source=sources[0], target=targets[0], allocation=actual_alloc, node_registry=node_registry, k=3)